# DTD Caption Experiment C: Frozen Florence + Image-Grounded Short Spans

This notebook creates a **third caption approach** for comparison with the original Florence CSV and the
decoder-LoRA CSV. It does **not** fine-tune Florence, CLIP or M-TACS. No annotation targets or PEFT wheels are needed.
It uses the same local Microsoft Florence-2-base and local full CLIP ViT-B/32 checkpoint already attached.

For each image, Florence produces its original greedy description plus three beam-search descriptions.
The notebook extracts existing 5-8-word spans within sentences, avoids some incomplete openings/endings,
and chooses a span by **CLIP image-to-text cosine similarity to that image**. It never creates a generic replacement
caption to hide a failure. If no candidate is eligible, it explicitly falls back to the original prefix and records it.
CLIP ranking can prefer generic/object-oriented or misleading spans; it is a hypothesis, not guaranteed improvement.

**Generation inputs are pixels plus the fixed Florence task token only.** No filename, category, correct class,
reference caption or test classification result is supplied to generation. The global class vocabulary is used
only after generation for the same `dtd_lexical_v2` eligibility checks and diagnostic text-to-class scores.
Text-to-class scores do not select the winning caption. Alias checks are diagnostic, not a stricter hidden filter.

The notebook uses official DTD split 1 (1,880 train / 1,880 val / 1,880 test). There is no classifier training here.
For downstream comparison, use identical M-TACS architecture, split, pool, seed, batch size and 50-epoch budget.
Do not select the caption method or change its settings based on repeated test F1.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import json
import shutil
import subprocess
import sys
from pathlib import Path
from importlib.util import find_spec

required = [name for name in ("torch", "numpy", "PIL", "pandas") if find_spec(name) is None]
if required:
    raise RuntimeError("Missing offline packages: " + ", ".join(required))
print("Offline preflight passed. Transformers is loaded only in the isolated legacy worker; workers=0.")


In [ ]:
DTD_ROOT = Path(os.getenv("DTD_ROOT", "/kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd"))
DTD_SPLIT = int(os.getenv("DTD_SPLIT", "1"))
PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_DIR = Path(os.getenv("DTD_GROUNDED_CAPTION_OUTPUT", str(
    PROJECT_DIR / "outputs" / f"florence2_dtd_split{DTD_SPLIT}_grounded_caption_v1")))
FLORENCE_MODEL_REF = os.getenv("FLORENCE_MODEL_REF", "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base")
OFFLINE_ASSETS_ROOT = os.getenv("DTD_OFFLINE_ASSETS_ROOT", "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi")
CLIP_MODEL_LOCAL_PATH = os.getenv("CLIP_MODEL_LOCAL_PATH", "/kaggle/input/datasets/nguyenducxu/vit-32/model_clip_vit")
SETTINGS = {"batch_size": 8, "num_beams": 3, "num_return_sequences": 3,
    "max_new_tokens": 48, "max_spans_per_hypothesis": 48, "clip_text_batch_size": 256}
SEED = 42
MAX_NEW_IMAGES = 0  # Smoke test: set 32; then return to 0 and rerun to finish all missing images.

# Optional exact attached paths. Never auto-select another file called captions.csv.
OLD_CAPTIONS_CSV = os.getenv("DTD_ORIGINAL_CAPTIONS_CSV", "")
LORA_CAPTIONS_CSV = os.getenv("DTD_LORA_CAPTIONS_CSV", "")
WRITE_MATCHED_MASK_CSVS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("New experiment output:", OUTPUT_DIR)


## Four Outputs to Explain Differences

| CSV | Florence proposals | Short-text choice | Purpose |
|---|---|---|---|
| `controls/prefix_control/captions.csv` | One greedy description | Original cleanup / first 8 words | Fresh procedural baseline |
| `controls/first_valid_control/captions.csv` | Same greedy description | First eligible span, no CLIP ranking | Effect of short-span extraction |
| `controls/single_grounded_control/captions.csv` | Same greedy description | Image-CLIP ranking of the same span pool | Effect of image-grounded ranking |
| `exports/captions.csv` | Greedy + 3 beam hypotheses | Same extraction + image-CLIP ranking | Main new caption approach |

First-valid vs single-grounded uses the same candidates and availability mask. Single-grounded vs main changes
the proposal pool. These are diagnostics you may run later; initially compare the main CSV to the two existing CSVs.
The fresh prefix control is not claimed byte-identical to a historical CSV: attached assets/runtime can differ.
Attach the actual old CSVs via the configuration above for an exact file-level comparison.
Three beam hypotheses are **caption alternatives for one image**, not three selected context images.
Downstream M-TACS still uses **K=1** in the matched experiment.

The outer policy stays `dtd_lexical_v2` (5-8 words, canonical class words screened, CLIP class score diagnostic only).
Selection can change validity/coverage by finding other eligible spans. Optional matched-mask CSVs later hold
availability fixed so changed coverage is not confused with changed text content. None of this certifies no semantic leakage.


In [ ]:
"""Frozen Florence proposals and image-grounded short-span selection for DTD."""
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

def clean_dtd_caption(text):
    import re
    text = str(text).strip()
    for _ in range(3):
        before = text
        text = re.sub(r"^(?:(?:the|this)\s+)?(?:image|picture|photo(?:graph)?)\s+"
                      r"(?:is|shows|depicts|features|contains)\s+", "", text, flags=re.I)
        text = re.sub(r"^(?:(?:a|an|the)\s+)?close[ -]up(?:\s+view)?\s+(?:of|showing)\s+",
                      "", text, flags=re.I)
        text = re.sub(r"^(?:a|an|the)(?:\s+|$)", "", text, flags=re.I)
        if text == before:
            break
    words = re.findall(r"[A-Za-z0-9]+(?:['\u2019][A-Za-z0-9]+)?", text)
    selected = []
    normalized_count = 0
    for word in words:
        count = len(re.findall(r"[A-Za-z0-9]+", word))
        if normalized_count + count > 8:
            break
        selected.append(word)
        normalized_count += count
    return " ".join(selected)
def atomic_csv(path, rows, columns):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)

def checkpoint_asset_identity(model_dir):
    model_dir = Path(model_dir)
    files = sorted(path for path in model_dir.iterdir() if path.is_file() and
                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))
    if not any(path.suffix in {".safetensors", ".bin"} for path in files):
        raise ValueError("Checkpoint has no weight files")
    return {path.name: file_sha256(path) for path in files}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def load_caption_manifest(root, split, expected_classes=47, per_class=40):
    root = Path(root).expanduser().resolve()
    if not 1 <= int(split) <= 10:
        raise ValueError("DTD_SPLIT must be in 1..10")
    images = (root / "images").resolve()
    names = sorted(path.name for path in images.iterdir() if path.is_dir())
    if len(names) != expected_classes:
        raise ValueError(f"Expected {expected_classes} DTD image folders; found {len(names)} at {images}")
    records, seen = [], set()
    for role in ("train", "val", "test"):
        counts = defaultdict(int)
        for raw in (root / "labels" / f"{role}{split}.txt").read_text(encoding="utf-8-sig").splitlines():
            text = raw.strip().replace("\\", "/")
            if not text:
                continue
            parts = PurePosixPath(text).parts
            if len(parts) != 2 or parts[0] not in names or any(part in {"..", "."} for part in parts):
                raise ValueError(f"Unexpected official DTD path: {text!r}")
            image_id = PurePosixPath(*parts).as_posix()
            path = images.joinpath(*parts).resolve()
            if not path.is_relative_to(images) or not path.is_file():
                raise FileNotFoundError(f"DTD image missing/outside images root: {path}")
            if image_id in seen:
                raise ValueError(f"Duplicate image or overlapping official splits: {image_id}")
            seen.add(image_id)
            counts[parts[0]] += 1
            records.append({"image_id": image_id, "image_path": str(path), "split": role})
        if set(counts) != set(names) or any(count != per_class for count in counts.values()):
            raise ValueError(f"Official {role}{split}.txt must contain {per_class} images/class")
    return records, names

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,
                                    allow_nan=False).encode()).hexdigest()

def read_csv_rows(path, required=()):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):
            raise ValueError(f"Missing/duplicate CSV header: {path}")
        if not set(required).issubset(reader.fieldnames):
            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")
        rows = list(reader)
    if any(None in row or any(value is None for value in row.values()) for row in rows):
        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")
    return rows


TASK = "<MORE_DETAILED_CAPTION>"
METHOD_VERSION = "florence_dtd_grounded_spans_v1"
METHODS = ("prefix_control", "first_valid_control", "single_grounded_control", "multibeam_grounded")
BASE_COLUMNS = ["image_id", "split", "raw_caption", "final_caption", "caption_valid", "leakage_detected",
    "lexical_leakage_detected", "semantic_leakage_detected", "semantic_leakage_score",
    "semantic_leakage_class", "retry_count", "vlm_model", "prompt_version"]
COLUMNS = BASE_COLUMNS + ["caption_input_policy", "semantic_audit_status", "captioner_fingerprint",
    "generation_method", "source_long_caption", "source_hypothesis", "source_sentence", "source_word_start",
    "source_word_end", "image_text_similarity", "alias_hit_diagnostic", "selection_fallback"]
EDGE_WORDS = set("a an the this that these those it its is are was were be been being and or but with of to in on at by for from into as has have had can will which there s t".split())
BAD_START = EDGE_WORDS | {"image", "photo", "picture", "photograph", "shows", "showing", "depicts"}
# Same diagnostic vocabulary as the existing DTD ablations; never a new eligibility gate.
ALIASES = {
    "banded": ["bands", "banding"], "blotchy": ["blotches", "blotch"], "braided": ["braid", "braids"],
    "bubbly": ["bubbles", "bubble"], "bumpy": ["bumps", "bump"], "chequered": ["checkered", "checkerboard"],
    "cobwebbed": ["cobwebs", "cobweb"], "cracked": ["cracks", "crack"],
    "crosshatched": ["crosshatched", "cross hatching"], "crystalline": ["crystals", "crystal"],
    "dotted": ["dots", "polka dots"], "fibrous": ["fibers", "fibres"], "flecked": ["flecks"],
    "freckled": ["freckles"], "frilly": ["frills"], "gauzy": ["gauze"], "gridded": ["grid", "grids"],
    "grooved": ["grooves"], "honeycombed": ["honeycomb"], "interlaced": ["interlacing"],
    "knitted": ["knit", "knitting"], "lacelike": ["lace", "lace like"], "lined": ["parallel lines"],
    "marbled": ["marbling"], "meshed": ["mesh"], "paisley": [], "perforated": ["perforations"],
    "pitted": ["pits"], "pleated": ["pleats"], "polka-dotted": ["polka dotted", "polka dots"],
    "porous": ["pores"], "potholed": ["potholes"], "scaly": ["scales"], "smeared": ["smears"],
    "spiralled": ["spiraled", "spirals", "spiral"], "sprinkled": ["sprinkles"], "stained": ["stains"],
    "stratified": ["strata"], "striped": ["stripes", "stripe"], "studded": ["studs"],
    "swirly": ["swirls", "swirl"], "veined": ["veins"], "waffled": ["waffle"],
    "woven": ["weave", "weaving"], "wrinkled": ["wrinkles"], "zigzagged": ["zigzag", "zig zag"],
}


def normalize(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).replace("_", " ").lower()).strip()


def bool_value(value):
    return str(value).lower() in {"true", "yes", "1"}


def term_hits(text, names, aliases=False):
    padded = " " + normalize(text) + " "
    terms = set(normalize(name) for name in names)
    if aliases:
        terms.update(normalize(term) for name in names for term in ALIASES.get(name, []))
    return sorted(term for term in terms if term and " " + term + " " in padded)


def valid_short(text, names):
    return 5 <= len(normalize(text).split()) <= 8 and not term_hits(text, names)


def strip_opening(text):
    for _ in range(3):
        before = text
        text = re.sub(r"^(?:(?:the|this)\s+)?(?:image|picture|photo(?:graph)?)\s+"
                      r"(?:is|shows|depicts|features|contains)\s+", "", text, flags=re.I)
        text = re.sub(r"^(?:(?:a|an|the)\s+)?close[ -]up(?:\s+view)?\s+(?:of|showing)\s+", "", text, flags=re.I)
        text = re.sub(r"^(?:a|an|the)(?:\s+|$)", "", text, flags=re.I)
        if text == before:
            break
    return text.strip()


def candidate_spans(raw, names, limit=48, hypothesis=0):
    """Extract substrings only; do not invent replacements or cross sentence boundaries."""
    candidates, seen = [], set()
    sentences = re.split(r"(?<=[.!?;:])\s+", str(raw).strip())
    for sentence_index, sentence in enumerate(sentences):
        words = re.findall(r"[A-Za-z0-9]+(?:['\u2019][A-Za-z0-9]+)?", strip_opening(sentence))
        for start in range(len(words)):
            for length in (8, 7, 6, 5):
                selected = words[start:start + length]
                if len(selected) != length:
                    continue
                text = " ".join(selected)
                key = normalize(text)
                if (key in seen or normalize(selected[0]) in BAD_START or normalize(selected[-1]) in EDGE_WORDS
                        or not valid_short(text, names) or clean_dtd_caption(text) != text):
                    continue
                tokens = key.split()
                if len(set(tokens)) < 4 or max(Counter(tokens).values()) > 2:
                    continue
                seen.add(key)
                candidates.append({"text": text, "long_caption": raw, "hypothesis": hypothesis,
                    "sentence": sentence_index, "word_start": start, "word_end": start + length})
    if len(candidates) > limit:
        indices = [round(i * (len(candidates) - 1) / (limit - 1)) for i in range(limit)] if limit > 1 else [0]
        candidates = [candidates[index] for index in indices]
    return candidates


def proposal_sets(raw_proposals, names, span_limit):
    prefix = {"text": clean_dtd_caption(raw_proposals[0]), "long_caption": raw_proposals[0],
              "hypothesis": 0, "sentence": -1, "word_start": 0, "word_end": -1}
    all_spans, seen = [], set()
    single = candidate_spans(raw_proposals[0], names, span_limit, 0)
    for index, raw in enumerate(raw_proposals):
        for candidate in candidate_spans(raw, names, span_limit, index):
            key = normalize(candidate["text"])
            if key not in seen:
                seen.add(key)
                all_spans.append(candidate)
    return prefix, single, all_spans


def select_candidate(candidates, scores, fallback):
    if not candidates:
        return fallback, True
    # Stable ordering for exact ties; class-audit scores are deliberately NOT consulted.
    return max(candidates, key=lambda candidate: round(scores[candidate["text"]]["image_cosine"], 7)), False


def row_for(record, selected, score, method, identity, names, fallback=False):
    text = selected["text"]
    if clean_dtd_caption(text) != text:
        raise ValueError("Export would change text when M-TACS cleans raw_caption; inspect this candidate")
    lexical = bool(term_hits(text, names))
    semantic = score.get("class_cosine", "") if score else ""
    return {"image_id": record["image_id"], "split": record["split"], "raw_caption": text, "final_caption": text,
        "caption_valid": valid_short(text, names), "leakage_detected": lexical,
        "lexical_leakage_detected": lexical, "semantic_leakage_detected": semantic != "" and semantic >= .30,
        "semantic_leakage_score": semantic, "semantic_leakage_class": score.get("class_name", "") if score else "",
        "retry_count": 0, "vlm_model": "microsoft/Florence-2-base+frozen-grounded-selection:" + identity[:16],
        "prompt_version": METHOD_VERSION, "caption_input_policy": "dtd_lexical_v2",
        "semantic_audit_status": "diagnostic_only" if score else "not_run_empty_caption",
        "captioner_fingerprint": identity, "generation_method": method,
        "source_long_caption": selected["long_caption"], "source_hypothesis": selected["hypothesis"],
        "source_sentence": selected["sentence"], "source_word_start": selected["word_start"],
        "source_word_end": selected["word_end"], "image_text_similarity": score.get("image_cosine", "") if score else "",
        "alias_hit_diagnostic": bool(term_hits(text, names, aliases=True)), "selection_fallback": fallback}


def build_image_result(record, proposals, scores, names, settings, identity):
    prefix, single, multiple = proposal_sets(proposals, names, settings["max_spans_per_hypothesis"])
    one_best, one_fallback = select_candidate(single, scores, prefix)
    multi_best, multi_fallback = select_candidate(multiple, scores, prefix)
    selected = {"prefix_control": (prefix, False), "first_valid_control": (single[0] if single else prefix, not single),
                "single_grounded_control": (one_best, one_fallback), "multibeam_grounded": (multi_best, multi_fallback)}
    rows = {method: row_for(record, candidate, scores.get(candidate["text"]), method, identity, names, fallback)
            for method, (candidate, fallback) in selected.items()}
    audit = [{**candidate, **scores[candidate["text"]],
              "alias_hits": term_hits(candidate["text"], names, aliases=True)} for candidate in multiple]
    return {"identity": identity, "image_id": record["image_id"], "split": record["split"],
            "proposals": proposals, "rows": rows, "candidate_audit": audit}


def load_rgb(path):
    from PIL import Image, ImageOps
    with Image.open(path) as source:
        return ImageOps.exif_transpose(source).convert("RGB")


def model_autocast(device):
    import torch
    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")


class FrozenFlorence:
    def __init__(self, path, device):
        import torch
        from transformers import AutoModelForCausalLM, AutoProcessor
        self.device = device
        self.dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else torch.float32
        self.processor = AutoProcessor.from_pretrained(path, trust_remote_code=True, local_files_only=True)
        self.model = AutoModelForCausalLM.from_pretrained(path, trust_remote_code=True, local_files_only=True,
            torch_dtype=self.dtype, low_cpu_mem_usage=True, attn_implementation="eager").to(device).eval()
        self.model.requires_grad_(False)

    def generate(self, images, beams, returns, tokens):
        import torch
        inputs = self.processor(text=[TASK] * len(images), images=images, padding=True, return_tensors="pt")
        inputs = {key: value.to(device=self.device, dtype=self.dtype if key == "pixel_values" else value.dtype)
                  for key, value in inputs.items() if torch.is_tensor(value)}
        with torch.inference_mode(), model_autocast(self.device):
            output = self.model.generate(**inputs, max_new_tokens=tokens, num_beams=beams,
                num_return_sequences=returns, do_sample=False)
        decoded = self.processor.batch_decode(output, skip_special_tokens=False)
        if len(decoded) != len(images) * returns:
            raise ValueError("Wrong Florence beam output count; no batch rows were cached")
        results = []
        for index, image in enumerate(images):
            group = []
            for text in decoded[index * returns:(index + 1) * returns]:
                parsed = self.processor.post_process_generation(text, task=TASK, image_size=(image.width, image.height))
                group.append(str(parsed.get(TASK, text) if isinstance(parsed, dict) else text).strip())
            results.append(group)
        return results


class ImageTextRanker:
    def __init__(self, path, names, device, text_batch_size=256):
        import torch
        from transformers import AutoTokenizer, CLIPImageProcessor, CLIPModel
        self.device, self.names, self.text_batch_size = device, names, text_batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
        self.model, loading = CLIPModel.from_pretrained(path, local_files_only=True, output_loading_info=True)
        missing = [key for key in loading.get("missing_keys", []) if not key.endswith("position_ids")]
        if missing or loading.get("mismatched_keys"):
            raise ValueError(f"A complete CLIP image AND text checkpoint is required; missing weights: {missing[:8]}")
        self.model = self.model.to(device).eval()
        self.model.requires_grad_(False)
        if (Path(path) / "preprocessor_config.json").is_file():
            self.image_processor = CLIPImageProcessor.from_pretrained(path, local_files_only=True)
            self.preprocessing_source = "checkpoint_preprocessor_config"
        else:
            size = self.model.config.vision_config.image_size
            if size != 224 or self.model.config.vision_config.patch_size != 32:
                raise ValueError("Missing CLIP image preprocessing config for a non-ViT-B/32 checkpoint")
            self.image_processor = CLIPImageProcessor()
            self.preprocessing_source = "transformers_4_49_standard_clip_vit_b32_defaults"
            print("CLIP preprocessor_config.json absent: using recorded standard CLIP ViT-B/32 preprocessing defaults", flush=True)
        prompts = [template.format(name=name.replace("_", " ").replace("-", " ")) for name in names
                   for template in ("a photo of {name}", "a scene showing {name}")]
        vectors = self.encode_text(prompts).reshape(len(names), 2, -1).mean(1)
        self.class_vectors = torch.nn.functional.normalize(vectors, dim=1)

    def encode_text(self, texts):
        import torch
        result = []
        with torch.inference_mode():
            for start in range(0, len(texts), self.text_batch_size):
                tokens = self.tokenizer(texts[start:start + self.text_batch_size], padding=True,
                    truncation=True, max_length=self.model.config.text_config.max_position_embeddings, return_tensors="pt")
                tokens = {key: value.to(self.device) for key, value in tokens.items()}
                with model_autocast(self.device):
                    features = self.model.get_text_features(**tokens)
                result.append(torch.nn.functional.normalize(features.float(), dim=1))
        return torch.cat(result)

    def score(self, images, text_groups):
        import torch
        unique = list(dict.fromkeys(text for group in text_groups for text in group if text.strip()))
        if not unique:
            return [{} for _ in images]
        texts = self.encode_text(unique)
        positions = {text: index for index, text in enumerate(unique)}
        pixels = self.image_processor(images=images, return_tensors="pt")["pixel_values"].to(self.device)
        with torch.inference_mode(), model_autocast(self.device):
            features = self.model.get_image_features(pixel_values=pixels)
        features = torch.nn.functional.normalize(features.float(), dim=1)
        image_scores = features @ texts.T
        class_scores, class_indices = (texts @ self.class_vectors.T).max(1)
        image_scores, class_scores, class_indices = image_scores.cpu().tolist(), class_scores.cpu().tolist(), class_indices.cpu().tolist()
        result = []
        for i, group in enumerate(text_groups):
            result.append({text: {"image_cosine": image_scores[i][positions[text]],
                "class_cosine": class_scores[positions[text]],
                "class_name": self.names[class_indices[positions[text]]]} for text in group if text.strip()})
        return result


def validate_settings(settings):
    for key in ("batch_size", "num_beams", "num_return_sequences", "max_new_tokens", "max_spans_per_hypothesis", "clip_text_batch_size"):
        if type(settings[key]) is not int or settings[key] < 1:
            raise ValueError(f"{key} must be a positive integer")
    if settings["num_beams"] < 2 or settings["num_return_sequences"] > settings["num_beams"]:
        raise ValueError("Use beam search with 1 <= num_return_sequences <= num_beams and num_beams >= 2")


def cache_path(output, image_id):
    return Path(output) / "image_cache" / (object_hash(image_id) + ".json")


def read_cached(output, record, identity):
    path = cache_path(output, record["image_id"])
    if not path.is_file():
        return None
    cached = json.loads(path.read_text(encoding="utf-8"))
    if cached["identity"] != identity or cached["image_id"] != record["image_id"] or cached["split"] != record["split"]:
        raise ValueError("Wrong experiment/image in cache; original cache was not modified")
    if set(cached["rows"]) != set(METHODS):
        raise ValueError("Incomplete per-image transaction in cache")
    for method, row in cached["rows"].items():
        if (set(row) != set(COLUMNS) or row["image_id"] != record["image_id"] or row["split"] != record["split"]
                or row["captioner_fingerprint"] != identity or row["generation_method"] != method):
            raise ValueError("Invalid cached row schema/identity")
    return cached


def run_manifest(payload):
    import torch
    import transformers
    validate_settings(payload["settings"])
    records = payload["records"]
    if not records or len({r["image_id"] for r in records}) != len(records):
        raise ValueError("Missing/duplicate dataset records")
    return {"version": METHOD_VERSION, "task": TASK, "settings": payload["settings"], "seed": payload["seed"],
        "code_sha256": payload["code_sha256"], "dtd_split": payload["dtd_split"],
        "class_names_for_post_generation_audit": payload["class_names"],
        "base_florence_assets": checkpoint_asset_identity(payload["florence"]),
        "clip_assets": checkpoint_asset_identity(payload["clip"]),
        "image_preprocessing": "checkpoint_config_if_present_else_standard_clip_vit_b32_defaults",
        "images": [{"image_id": r["image_id"], "split": r["split"], "sha256": file_sha256(r["image_path"])} for r in records],
        "runtime": {"torch": torch.__version__, "transformers": transformers.__version__,
            "bf16": torch.cuda.is_available() and torch.cuda.is_bf16_supported()},
        "canonical_policy": "dtd_lexical_v2", "semantic_audit_threshold": .30,
        "semantic_hard_gate": False, "trainable_parameters": 0}


def export_cached(payload, identity):
    output = Path(payload["output"])
    cached = [read_cached(output, r, identity) for r in payload["records"]]
    complete = all(item is not None for item in cached)
    available = [item for item in cached if item is not None]
    folder = output if complete else output / "preview_incomplete"
    paths = {}
    for method in METHODS:
        path = folder / "exports" / "captions.csv" if method == "multibeam_grounded" else folder / "controls" / method / "captions.csv"
        atomic_csv(path, [item["rows"][method] for item in available], COLUMNS)
        paths[method] = str(path)
    audit_path = folder / "candidate_audit.jsonl"
    audit_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = audit_path.with_suffix(".jsonl.tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for item in available:
            handle.write(json.dumps({"image_id": item["image_id"], "split": item["split"],
                "raw_proposals": item["proposals"], "candidates": item["candidate_audit"]}, ensure_ascii=False, allow_nan=False) + "\n")
    temporary.replace(audit_path)
    summaries = []
    for method in METHODS:
        for role in ("train", "val", "test"):
            rows = [item["rows"][method] for item in available if item["split"] == role]
            if not rows:
                continue
            summaries.append({"method": method, "split": role, "images": len(rows),
                "valid": sum(row["caption_valid"] for row in rows),
                "valid_rate": sum(row["caption_valid"] for row in rows) / len(rows),
                "unique_captions": len({row["final_caption"] for row in rows}),
                "canonical_hits": sum(row["lexical_leakage_detected"] for row in rows),
                "alias_hits_diagnostic": sum(row["alias_hit_diagnostic"] for row in rows),
                "fallbacks": sum(row["selection_fallback"] for row in rows),
                "mean_image_text_score": sum(row["image_text_similarity"] for row in rows if row["image_text_similarity"] != "") /
                    max(1, sum(row["image_text_similarity"] != "" for row in rows))})
    if summaries:
        atomic_csv(folder / "caption_generation_summary.csv", summaries, list(summaries[0]))
    summary = {"identity": identity, "complete": complete, "rows": len(available), "expected": len(cached),
        "csv_paths": paths, "csv_sha256": {key: file_sha256(path) for key, path in paths.items()},
        "optimized_clip_score_is_not_independent_quality_evidence": True,
        "downstream_improvement_verified": False, "human_ground_truth": False}
    atomic_json(output / "generation_status.json", summary)
    return summary


def run_generation(payload, device, florence_factory=FrozenFlorence, ranker_factory=ImageTextRanker):
    import torch
    output = Path(payload["output"])
    manifest = run_manifest(payload)
    identity = object_hash(manifest)
    manifest_path = output / "generation_manifest.json"
    if manifest_path.is_file() and json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise ValueError("Changed assets/code/dataset/settings/runtime: choose a new output directory")
    if not manifest_path.is_file() and ((output / "image_cache").exists() or (output / "exports").exists()):
        raise ValueError("Existing generated artifacts have no matching manifest; use another output directory")
    atomic_json(manifest_path, manifest)
    pending = [r for r in payload["records"] if read_cached(output, r, identity) is None]
    limit = payload.get("max_new_images", 0)
    if limit < 0:
        raise ValueError("max_new_images must be >= 0; zero processes all missing images")
    if limit:
        pending = pending[:limit]
    print(f"Grounded captions: remaining this run={len(pending)}; output={output}", flush=True)
    if pending:
        generator = florence_factory(payload["florence"], device)
        ranker = ranker_factory(payload["clip"], payload["class_names"], device, payload["settings"]["clip_text_batch_size"])
        atomic_json(output / "clip_preprocessing_used.json", {"source": ranker.preprocessing_source,
                    "config": ranker.image_processor.to_dict()})
        size, offset = payload["settings"]["batch_size"], 0
        while offset < len(pending):
            group = pending[offset:offset + size]
            try:
                images = [load_rgb(r["image_path"]) for r in group]
                greedy = generator.generate(images, 1, 1, payload["settings"]["max_new_tokens"])
                beams = generator.generate(images, payload["settings"]["num_beams"], payload["settings"]["num_return_sequences"], payload["settings"]["max_new_tokens"])
                if (len(greedy) != len(group) or len(beams) != len(group) or any(len(p) != 1 for p in greedy)
                        or any(len(p) != payload["settings"]["num_return_sequences"] for p in beams)):
                    raise ValueError("Wrong image/hypothesis count from Florence")
                proposals = [one + many for one, many in zip(greedy, beams)]
                text_groups = []
                for raw in proposals:
                    prefix, single, multiple = proposal_sets(raw, payload["class_names"], payload["settings"]["max_spans_per_hypothesis"])
                    text_groups.append(list(dict.fromkeys([prefix["text"]] + [item["text"] for item in multiple])))
                scores = ranker.score(images, text_groups)
                if len(scores) != len(group) or any(set(score) != {t for t in texts if t.strip()} for score, texts in zip(scores, text_groups)):
                    raise ValueError("Wrong CLIP score count/keys; no batch rows were cached")
                if any(not math.isfinite(value["image_cosine"]) or not math.isfinite(value["class_cosine"])
                       for score in scores for value in score.values()):
                    raise ValueError("Non-finite CLIP score")
                transactions = [build_image_result(r, p, s, payload["class_names"], payload["settings"], identity)
                                for r, p, s in zip(group, proposals, scores)]
            except torch.cuda.OutOfMemoryError:
                if len(group) == 1:
                    raise
                size = max(1, len(group) // 2)
                torch.cuda.empty_cache()
                print("Caption batch reduced after CUDA OOM to", size, flush=True)
                continue
            for record, result in zip(group, transactions):
                atomic_json(cache_path(output, record["image_id"]), result)
            offset += len(group)
            print(f"Completed {offset}/{len(pending)} newly queued images", flush=True)
    return export_cached(payload, identity)


def worker_main(payload_path):
    import torch
    import transformers
    if not transformers.__version__.startswith("4.49."):
        raise RuntimeError("Use the isolated Transformers 4.49.x worker for the attached legacy Florence model")
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator for this Florence + full CLIP generation worker")
    payload = json.loads(Path(payload_path).read_text(encoding="utf-8"))
    torch.manual_seed(payload["seed"])
    torch.cuda.manual_seed_all(payload["seed"])
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    summary = run_generation(payload, torch.device("cuda"))
    print(json.dumps(summary, indent=2), flush=True)


def effective_dtd_v2(row, names):
    text = clean_dtd_caption(str(row.get("raw_caption", "")).strip() or row["final_caption"])
    try:
        finite_score = math.isfinite(float(row["semantic_leakage_score"]))
    except (TypeError, ValueError):
        finite_score = False
    recoverable = bool_value(row["caption_valid"]) or (bool_value(row["semantic_leakage_detected"]) and finite_score)
    return text, bool(valid_short(text, names) and recoverable)


def compare_caption_files(paths, records, names, output_dir, write_matched_masks=True):
    """Read existing CSVs without changing them; compare the actual dtd_lexical_v2 model inputs."""
    expected = {r["image_id"]: r["split"] for r in records}
    if not paths or len(expected) != len(records):
        raise ValueError("Provide named complete CSVs and distinct dataset records")
    methods, source_hashes = {}, {}
    for label, path in paths.items():
        if not re.fullmatch(r"[a-z0-9_]+", label):
            raise ValueError("Comparison labels must be lowercase letters/digits/underscores")
        rows = read_csv_rows(path, ["image_id", "split", "final_caption", "caption_valid", "leakage_detected",
                                    "semantic_leakage_detected", "semantic_leakage_score"])
        mapped = {row["image_id"]: row for row in rows}
        if len(mapped) != len(rows) or set(mapped) != set(expected) or any(row["split"] != expected[key] for key, row in mapped.items()):
            raise ValueError(f"{label}: incomplete, duplicate or wrong official DTD split; original CSV was not modified")
        methods[label] = {key: {"row": mapped[key], "text": effective_dtd_v2(mapped[key], names)[0],
                                "valid": effective_dtd_v2(mapped[key], names)[1]} for key in expected}
        source_hashes[label] = file_sha256(path)
    output_dir = Path(output_dir)
    signatures = {"csv_sha256": source_hashes, "policy": "dtd_lexical_v2", "names": names,
                  "image_ids_and_splits": expected, "comparison_version": 1}
    lock = output_dir / "comparison_manifest.json"
    if lock.exists() and json.loads(lock.read_text(encoding="utf-8")) != signatures:
        raise ValueError("Comparison inputs changed; choose a new comparison output directory")
    atomic_json(lock, signatures)
    common = {key: all(method[key]["valid"] for method in methods.values()) for key in expected}
    summaries, details = [], []
    first_label = next(iter(methods))
    for label, method in methods.items():
        for role in ("train", "val", "test"):
            entries = [entry for key, entry in method.items() if expected[key] == role]
            if entries:
                summaries.append({"method": label, "split": role, "images": len(entries),
                    "valid": sum(entry["valid"] for entry in entries),
                    "valid_rate": sum(entry["valid"] for entry in entries) / len(entries),
                    "unique_input_texts": len({entry["text"] for entry in entries}),
                    "valid_with_alias_hit": sum(entry["valid"] and bool(term_hits(entry["text"], names, True)) for entry in entries),
                    "common_valid": sum(common[key] for key in expected if expected[key] == role),
                    "text_changed_vs_first": sum(method[key]["text"] != methods[first_label][key]["text"] for key in expected if expected[key] == role)})
        for key, entry in method.items():
            details.append({"method": label, "image_id": key, "split": expected[key], "model_input_text": entry["text"],
                            "valid": entry["valid"], "common_valid": common[key]})
        if write_matched_masks:
            masked = []
            for record in records:
                key = record["image_id"]
                row = dict(method[key]["row"])
                row.update(original_caption_valid=row["caption_valid"], original_semantic_flag=row["semantic_leakage_detected"],
                    original_semantic_score=row["semantic_leakage_score"], caption_valid=common[key],
                    caption_input_policy="dtd_lexical_v2", comparison_mask="common_valid_intersection")
                if not common[key]:
                    # Prevent legacy semantic-only recovery from switching a deliberately masked caption back on.
                    row.update(semantic_leakage_detected=False, semantic_leakage_score="",
                               semantic_audit_status="original_audit_retained_in_original_columns")
                masked.append(row)
            columns = list(dict.fromkeys(key for row in masked for key in row))
            atomic_csv(output_dir / "matched_masks" / label / "captions.csv", masked, columns)
            assert all(effective_dtd_v2(row, names)[1] == common[row["image_id"]] for row in masked)
    atomic_csv(output_dir / "caption_comparison_summary.csv", summaries, list(summaries[0]))
    atomic_csv(output_dir / "caption_comparison_details.csv", details, list(details[0]))
    atomic_csv(output_dir / "common_valid_mask.csv", [{"image_id": key, "split": role, "caption_valid": common[key]}
               for key, role in expected.items()], ["image_id", "split", "caption_valid"])
    return summaries


In [ ]:
caption_records, class_names = load_caption_manifest(DTD_ROOT, DTD_SPLIT)
validate_settings(SETTINGS)
for split in ("train", "val", "test"):
    print(split, sum(row["split"] == split for row in caption_records))
print("Total:", len(caption_records), "| classes:", len(class_names))


## Offline Runtime and Assets

The attached Microsoft checkpoint is **legacy Florence**: use the same isolated Transformers **4.49.x** worker
as the earlier notebooks. A kernel with Transformers 5.x is fine; it is not downgraded or restarted.
Missing legacy packages are installed into this experiment's worker folder using attached offline wheels only.
There are no Internet/API requests, PEFT imports or DataLoader subprocesses.

This approach requires **both CLIP towers**, not a text-only checkpoint. Missing image/text weights are fatal,
not silently initialized. If CLIP's `preprocessor_config.json` is absent, the worker permits only standard
224px/patch32 CLIP and records use of Transformers 4.49's standard CLIPImageProcessor defaults. Otherwise it
uses the attached preprocessing config. Inspect `clip_preprocessing_used.json` when comparing asset provenance.

The protocol hashes images, model assets, code, settings and runtime versions. Resume keeps per-image atomic
transactions, so an interrupted run does not need to regenerate completed images. Changed protocols must use
a new output directory. Batch OOM halves the effective generation batch; other errors stop visibly.


In [ ]:
def florence_checkpoint(model_ref, roots):
    def checked(path):
        path = Path(path)
        try:
            config = json.loads((path / "config.json").read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return None
        if config.get("model_type") != "florence2":
            return None
        indexes = list(path.glob("*.index.json"))
        weights = any((path / name).is_file() for name in ("model.safetensors", "pytorch_model.bin"))
        for index in indexes:
            try:
                shards = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())
                weights |= bool(shards) and all((path / shard).is_file() for shard in shards)
            except (OSError, ValueError):
                continue
        if not weights:
            return None
        backend = "legacy" if "AutoModelForCausalLM" in config.get("auto_map", {}) else "native"
        if backend == "legacy" and not all((path / name).is_file() for name in (
                "configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py")):
            return None
        return path.resolve(), backend

    requested = Path(model_ref).expanduser()
    if requested.is_dir():
        result = checked(requested)
        if result:
            return result
        for config in sorted(requested.rglob("config.json")):
            result = checked(config.parent)
            if result:
                return result
        raise FileNotFoundError(f"Florence folder is incomplete: {requested}; need config, weights, processor/tokenizer and local model code.")
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git"})
            if len(Path(current).relative_to(root).parts) >= 8:
                dirs[:] = []
            if "config.json" in files:
                result = checked(current)
                if result:
                    return result
    raise FileNotFoundError(
        f"No complete local Florence-2 checkpoint found for {model_ref}. "
        "Attach notebook-mt-moi/Florence-2-base or set FLORENCE_MODEL_REF to the correct local folder. No download is attempted."
    )

def offline_caption_wheels(roots, destination):
    import zipfile
    destination = Path(destination)
    wheel_dirs = set()
    prefixes = ("transformers-", "tokenizers-", "huggingface_hub-", "accelerate-", "einops-", "timm-")
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git", "Florence-2-base"})
            if len(Path(current).relative_to(root).parts) >= 7:
                dirs[:] = []
            for name in sorted(files):
                path = Path(current) / name
                if name.endswith(".whl"):
                    wheel_dirs.add(str(path.parent))
                elif name.lower() in {"offline_wheels.zip", "wheels.zip"}:
                    with zipfile.ZipFile(path) as archive:
                        for member in archive.infolist():
                            filename = Path(member.filename.replace("\\", "/")).name
                            if not filename.endswith(".whl") or not filename.startswith(prefixes):
                                continue
                            destination.mkdir(parents=True, exist_ok=True)
                            target = destination / filename
                            if not target.is_file() or target.stat().st_size != member.file_size:
                                with archive.open(member) as source, target.open("wb") as handle:
                                    shutil.copyfileobj(source, handle)
                            wheel_dirs.add(str(destination))
    return sorted(wheel_dirs)

def caption_runtime_environment(backend, asset_root, runtime_dir):
    import subprocess
    import sys
    runtime_dir = Path(runtime_dir)
    runtime_dir.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ)
    env.update(TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
               HF_MODULES_CACHE=str(runtime_dir / "hf_modules"), PYTHONUNBUFFERED="1")
    target = runtime_dir / "legacy_packages"
    probe = (
        "import transformers; "
        + ("assert transformers.__version__.startswith('4.49.'), transformers.__version__; " if backend == "legacy" else "")
        + "from transformers import AutoProcessor, AutoTokenizer, CLIPModel; "
        + ("from transformers import AutoModelForCausalLM; " if backend == "legacy" else
           "from transformers import Florence2ForConditionalGeneration; ")
        + "import einops, timm, accelerate; print(transformers.__version__)"
    )
    if target.is_dir():
        env["PYTHONPATH"] = str(target) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    result = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
    if result.returncode == 0:
        print("Caption subprocess Transformers:", result.stdout.strip())
        return env
    if backend != "legacy":
        raise RuntimeError("The local native Florence checkpoint cannot load in this runtime. "
                           "Use a matching native Transformers installation or the attached legacy model + offline 4.49 wheels.\n"
                           + result.stderr[-3000:])
    wheel_dirs = offline_caption_wheels([Path(asset_root), Path("/kaggle/input")], runtime_dir / "wheels")
    if not wheel_dirs:
        raise FileNotFoundError(
            "The attached Microsoft Florence checkpoint needs Transformers 4.49.x. "
            f"No offline wheels found under {asset_root}. Attach offline_wheels.zip/wheels.zip or a wheel folder "
            "with transformers==4.49.0, tokenizers 0.21.x and huggingface-hub <1 compatible with this Python version. "
            "The notebook never enables Internet or modifies the kernel's Transformers."
        )
    command = [sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",
               "--no-deps", "--upgrade", "--target", str(target), "transformers==4.49.0",
               "tokenizers>=0.21,<0.22", "huggingface-hub>=0.26,<1"]
    import importlib.util
    for module in ("einops", "timm", "accelerate"):
        if importlib.util.find_spec(module) is None:
            command.append(module)
    for directory in wheel_dirs:
        command.extend(["--find-links", directory])
    print("Preparing isolated Florence dependencies from attached offline wheels; the notebook kernel is unchanged.")
    installed = subprocess.run(command, env=env, capture_output=True, text=True)
    if installed.returncode:
        raise RuntimeError("Offline Florence dependency setup failed. Wheels must match Kaggle's Python/platform.\n"
                           + installed.stdout[-2000:] + installed.stderr[-3000:])
    base_pythonpath = os.environ.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(target) + (os.pathsep + base_pythonpath if base_pythonpath else "")
    checked = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
    if checked.returncode:
        raise RuntimeError("The offline Florence environment still lacks a compatible dependency. "
                           "No kernel packages were changed.\n" + checked.stderr[-4000:])
    print("Isolated caption subprocess Transformers:", checked.stdout.strip())
    return env


def run_grounded_worker(worker_source, payload_path, runtime_dir, env):
    worker_path = Path(runtime_dir) / "generate_grounded_dtd.py"
    worker_path.write_text(worker_source, encoding="utf-8")
    log_path = Path(runtime_dir) / "generation.log"
    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, str(worker_path), str(payload_path)], env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
        finally:
            process.stdout.close()
    if code:
        raise RuntimeError(f"Caption worker failed; inspect {log_path}. Completed per-image caches are preserved for resume.")


GROUNDED_WORKER_SOURCE = '"""Frozen Florence proposals and image-grounded short-span selection for DTD."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport sys\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\ndef clean_dtd_caption(text):\n    import re\n    text = str(text).strip()\n    for _ in range(3):\n        before = text\n        text = re.sub(r"^(?:(?:the|this)\\s+)?(?:image|picture|photo(?:graph)?)\\s+"\n                      r"(?:is|shows|depicts|features|contains)\\s+", "", text, flags=re.I)\n        text = re.sub(r"^(?:(?:a|an|the)\\s+)?close[ -]up(?:\\s+view)?\\s+(?:of|showing)\\s+",\n                      "", text, flags=re.I)\n        text = re.sub(r"^(?:a|an|the)(?:\\s+|$)", "", text, flags=re.I)\n        if text == before:\n            break\n    words = re.findall(r"[A-Za-z0-9]+(?:[\'\\u2019][A-Za-z0-9]+)?", text)\n    selected = []\n    normalized_count = 0\n    for word in words:\n        count = len(re.findall(r"[A-Za-z0-9]+", word))\n        if normalized_count + count > 8:\n            break\n        selected.append(word)\n        normalized_count += count\n    return " ".join(selected)\ndef atomic_csv(path, rows, columns):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\ndef checkpoint_asset_identity(model_dir):\n    model_dir = Path(model_dir)\n    files = sorted(path for path in model_dir.iterdir() if path.is_file() and\n                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))\n    if not any(path.suffix in {".safetensors", ".bin"} for path in files):\n        raise ValueError("Checkpoint has no weight files")\n    return {path.name: file_sha256(path) for path in files}\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef load_caption_manifest(root, split, expected_classes=47, per_class=40):\n    root = Path(root).expanduser().resolve()\n    if not 1 <= int(split) <= 10:\n        raise ValueError("DTD_SPLIT must be in 1..10")\n    images = (root / "images").resolve()\n    names = sorted(path.name for path in images.iterdir() if path.is_dir())\n    if len(names) != expected_classes:\n        raise ValueError(f"Expected {expected_classes} DTD image folders; found {len(names)} at {images}")\n    records, seen = [], set()\n    for role in ("train", "val", "test"):\n        counts = defaultdict(int)\n        for raw in (root / "labels" / f"{role}{split}.txt").read_text(encoding="utf-8-sig").splitlines():\n            text = raw.strip().replace("\\\\", "/")\n            if not text:\n                continue\n            parts = PurePosixPath(text).parts\n            if len(parts) != 2 or parts[0] not in names or any(part in {"..", "."} for part in parts):\n                raise ValueError(f"Unexpected official DTD path: {text!r}")\n            image_id = PurePosixPath(*parts).as_posix()\n            path = images.joinpath(*parts).resolve()\n            if not path.is_relative_to(images) or not path.is_file():\n                raise FileNotFoundError(f"DTD image missing/outside images root: {path}")\n            if image_id in seen:\n                raise ValueError(f"Duplicate image or overlapping official splits: {image_id}")\n            seen.add(image_id)\n            counts[parts[0]] += 1\n            records.append({"image_id": image_id, "image_path": str(path), "split": role})\n        if set(counts) != set(names) or any(count != per_class for count in counts.values()):\n            raise ValueError(f"Official {role}{split}.txt must contain {per_class} images/class")\n    return records, names\n\ndef object_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,\n                                    allow_nan=False).encode()).hexdigest()\n\ndef read_csv_rows(path, required=()):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):\n            raise ValueError(f"Missing/duplicate CSV header: {path}")\n        if not set(required).issubset(reader.fieldnames):\n            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")\n        rows = list(reader)\n    if any(None in row or any(value is None for value in row.values()) for row in rows):\n        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")\n    return rows\n\n\nTASK = "<MORE_DETAILED_CAPTION>"\nMETHOD_VERSION = "florence_dtd_grounded_spans_v1"\nMETHODS = ("prefix_control", "first_valid_control", "single_grounded_control", "multibeam_grounded")\nBASE_COLUMNS = ["image_id", "split", "raw_caption", "final_caption", "caption_valid", "leakage_detected",\n    "lexical_leakage_detected", "semantic_leakage_detected", "semantic_leakage_score",\n    "semantic_leakage_class", "retry_count", "vlm_model", "prompt_version"]\nCOLUMNS = BASE_COLUMNS + ["caption_input_policy", "semantic_audit_status", "captioner_fingerprint",\n    "generation_method", "source_long_caption", "source_hypothesis", "source_sentence", "source_word_start",\n    "source_word_end", "image_text_similarity", "alias_hit_diagnostic", "selection_fallback"]\nEDGE_WORDS = set("a an the this that these those it its is are was were be been being and or but with of to in on at by for from into as has have had can will which there s t".split())\nBAD_START = EDGE_WORDS | {"image", "photo", "picture", "photograph", "shows", "showing", "depicts"}\n# Same diagnostic vocabulary as the existing DTD ablations; never a new eligibility gate.\nALIASES = {\n    "banded": ["bands", "banding"], "blotchy": ["blotches", "blotch"], "braided": ["braid", "braids"],\n    "bubbly": ["bubbles", "bubble"], "bumpy": ["bumps", "bump"], "chequered": ["checkered", "checkerboard"],\n    "cobwebbed": ["cobwebs", "cobweb"], "cracked": ["cracks", "crack"],\n    "crosshatched": ["crosshatched", "cross hatching"], "crystalline": ["crystals", "crystal"],\n    "dotted": ["dots", "polka dots"], "fibrous": ["fibers", "fibres"], "flecked": ["flecks"],\n    "freckled": ["freckles"], "frilly": ["frills"], "gauzy": ["gauze"], "gridded": ["grid", "grids"],\n    "grooved": ["grooves"], "honeycombed": ["honeycomb"], "interlaced": ["interlacing"],\n    "knitted": ["knit", "knitting"], "lacelike": ["lace", "lace like"], "lined": ["parallel lines"],\n    "marbled": ["marbling"], "meshed": ["mesh"], "paisley": [], "perforated": ["perforations"],\n    "pitted": ["pits"], "pleated": ["pleats"], "polka-dotted": ["polka dotted", "polka dots"],\n    "porous": ["pores"], "potholed": ["potholes"], "scaly": ["scales"], "smeared": ["smears"],\n    "spiralled": ["spiraled", "spirals", "spiral"], "sprinkled": ["sprinkles"], "stained": ["stains"],\n    "stratified": ["strata"], "striped": ["stripes", "stripe"], "studded": ["studs"],\n    "swirly": ["swirls", "swirl"], "veined": ["veins"], "waffled": ["waffle"],\n    "woven": ["weave", "weaving"], "wrinkled": ["wrinkles"], "zigzagged": ["zigzag", "zig zag"],\n}\n\n\ndef normalize(text):\n    return re.sub(r"[^a-z0-9]+", " ", str(text).replace("_", " ").lower()).strip()\n\n\ndef bool_value(value):\n    return str(value).lower() in {"true", "yes", "1"}\n\n\ndef term_hits(text, names, aliases=False):\n    padded = " " + normalize(text) + " "\n    terms = set(normalize(name) for name in names)\n    if aliases:\n        terms.update(normalize(term) for name in names for term in ALIASES.get(name, []))\n    return sorted(term for term in terms if term and " " + term + " " in padded)\n\n\ndef valid_short(text, names):\n    return 5 <= len(normalize(text).split()) <= 8 and not term_hits(text, names)\n\n\ndef strip_opening(text):\n    for _ in range(3):\n        before = text\n        text = re.sub(r"^(?:(?:the|this)\\s+)?(?:image|picture|photo(?:graph)?)\\s+"\n                      r"(?:is|shows|depicts|features|contains)\\s+", "", text, flags=re.I)\n        text = re.sub(r"^(?:(?:a|an|the)\\s+)?close[ -]up(?:\\s+view)?\\s+(?:of|showing)\\s+", "", text, flags=re.I)\n        text = re.sub(r"^(?:a|an|the)(?:\\s+|$)", "", text, flags=re.I)\n        if text == before:\n            break\n    return text.strip()\n\n\ndef candidate_spans(raw, names, limit=48, hypothesis=0):\n    """Extract substrings only; do not invent replacements or cross sentence boundaries."""\n    candidates, seen = [], set()\n    sentences = re.split(r"(?<=[.!?;:])\\s+", str(raw).strip())\n    for sentence_index, sentence in enumerate(sentences):\n        words = re.findall(r"[A-Za-z0-9]+(?:[\'\\u2019][A-Za-z0-9]+)?", strip_opening(sentence))\n        for start in range(len(words)):\n            for length in (8, 7, 6, 5):\n                selected = words[start:start + length]\n                if len(selected) != length:\n                    continue\n                text = " ".join(selected)\n                key = normalize(text)\n                if (key in seen or normalize(selected[0]) in BAD_START or normalize(selected[-1]) in EDGE_WORDS\n                        or not valid_short(text, names) or clean_dtd_caption(text) != text):\n                    continue\n                tokens = key.split()\n                if len(set(tokens)) < 4 or max(Counter(tokens).values()) > 2:\n                    continue\n                seen.add(key)\n                candidates.append({"text": text, "long_caption": raw, "hypothesis": hypothesis,\n                    "sentence": sentence_index, "word_start": start, "word_end": start + length})\n    if len(candidates) > limit:\n        indices = [round(i * (len(candidates) - 1) / (limit - 1)) for i in range(limit)] if limit > 1 else [0]\n        candidates = [candidates[index] for index in indices]\n    return candidates\n\n\ndef proposal_sets(raw_proposals, names, span_limit):\n    prefix = {"text": clean_dtd_caption(raw_proposals[0]), "long_caption": raw_proposals[0],\n              "hypothesis": 0, "sentence": -1, "word_start": 0, "word_end": -1}\n    all_spans, seen = [], set()\n    single = candidate_spans(raw_proposals[0], names, span_limit, 0)\n    for index, raw in enumerate(raw_proposals):\n        for candidate in candidate_spans(raw, names, span_limit, index):\n            key = normalize(candidate["text"])\n            if key not in seen:\n                seen.add(key)\n                all_spans.append(candidate)\n    return prefix, single, all_spans\n\n\ndef select_candidate(candidates, scores, fallback):\n    if not candidates:\n        return fallback, True\n    # Stable ordering for exact ties; class-audit scores are deliberately NOT consulted.\n    return max(candidates, key=lambda candidate: round(scores[candidate["text"]]["image_cosine"], 7)), False\n\n\ndef row_for(record, selected, score, method, identity, names, fallback=False):\n    text = selected["text"]\n    if clean_dtd_caption(text) != text:\n        raise ValueError("Export would change text when M-TACS cleans raw_caption; inspect this candidate")\n    lexical = bool(term_hits(text, names))\n    semantic = score.get("class_cosine", "") if score else ""\n    return {"image_id": record["image_id"], "split": record["split"], "raw_caption": text, "final_caption": text,\n        "caption_valid": valid_short(text, names), "leakage_detected": lexical,\n        "lexical_leakage_detected": lexical, "semantic_leakage_detected": semantic != "" and semantic >= .30,\n        "semantic_leakage_score": semantic, "semantic_leakage_class": score.get("class_name", "") if score else "",\n        "retry_count": 0, "vlm_model": "microsoft/Florence-2-base+frozen-grounded-selection:" + identity[:16],\n        "prompt_version": METHOD_VERSION, "caption_input_policy": "dtd_lexical_v2",\n        "semantic_audit_status": "diagnostic_only" if score else "not_run_empty_caption",\n        "captioner_fingerprint": identity, "generation_method": method,\n        "source_long_caption": selected["long_caption"], "source_hypothesis": selected["hypothesis"],\n        "source_sentence": selected["sentence"], "source_word_start": selected["word_start"],\n        "source_word_end": selected["word_end"], "image_text_similarity": score.get("image_cosine", "") if score else "",\n        "alias_hit_diagnostic": bool(term_hits(text, names, aliases=True)), "selection_fallback": fallback}\n\n\ndef build_image_result(record, proposals, scores, names, settings, identity):\n    prefix, single, multiple = proposal_sets(proposals, names, settings["max_spans_per_hypothesis"])\n    one_best, one_fallback = select_candidate(single, scores, prefix)\n    multi_best, multi_fallback = select_candidate(multiple, scores, prefix)\n    selected = {"prefix_control": (prefix, False), "first_valid_control": (single[0] if single else prefix, not single),\n                "single_grounded_control": (one_best, one_fallback), "multibeam_grounded": (multi_best, multi_fallback)}\n    rows = {method: row_for(record, candidate, scores.get(candidate["text"]), method, identity, names, fallback)\n            for method, (candidate, fallback) in selected.items()}\n    audit = [{**candidate, **scores[candidate["text"]],\n              "alias_hits": term_hits(candidate["text"], names, aliases=True)} for candidate in multiple]\n    return {"identity": identity, "image_id": record["image_id"], "split": record["split"],\n            "proposals": proposals, "rows": rows, "candidate_audit": audit}\n\n\ndef load_rgb(path):\n    from PIL import Image, ImageOps\n    with Image.open(path) as source:\n        return ImageOps.exif_transpose(source).convert("RGB")\n\n\ndef model_autocast(device):\n    import torch\n    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16\n    return torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")\n\n\nclass FrozenFlorence:\n    def __init__(self, path, device):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoProcessor\n        self.device = device\n        self.dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else torch.float32\n        self.processor = AutoProcessor.from_pretrained(path, trust_remote_code=True, local_files_only=True)\n        self.model = AutoModelForCausalLM.from_pretrained(path, trust_remote_code=True, local_files_only=True,\n            torch_dtype=self.dtype, low_cpu_mem_usage=True, attn_implementation="eager").to(device).eval()\n        self.model.requires_grad_(False)\n\n    def generate(self, images, beams, returns, tokens):\n        import torch\n        inputs = self.processor(text=[TASK] * len(images), images=images, padding=True, return_tensors="pt")\n        inputs = {key: value.to(device=self.device, dtype=self.dtype if key == "pixel_values" else value.dtype)\n                  for key, value in inputs.items() if torch.is_tensor(value)}\n        with torch.inference_mode(), model_autocast(self.device):\n            output = self.model.generate(**inputs, max_new_tokens=tokens, num_beams=beams,\n                num_return_sequences=returns, do_sample=False)\n        decoded = self.processor.batch_decode(output, skip_special_tokens=False)\n        if len(decoded) != len(images) * returns:\n            raise ValueError("Wrong Florence beam output count; no batch rows were cached")\n        results = []\n        for index, image in enumerate(images):\n            group = []\n            for text in decoded[index * returns:(index + 1) * returns]:\n                parsed = self.processor.post_process_generation(text, task=TASK, image_size=(image.width, image.height))\n                group.append(str(parsed.get(TASK, text) if isinstance(parsed, dict) else text).strip())\n            results.append(group)\n        return results\n\n\nclass ImageTextRanker:\n    def __init__(self, path, names, device, text_batch_size=256):\n        import torch\n        from transformers import AutoTokenizer, CLIPImageProcessor, CLIPModel\n        self.device, self.names, self.text_batch_size = device, names, text_batch_size\n        self.tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)\n        self.model, loading = CLIPModel.from_pretrained(path, local_files_only=True, output_loading_info=True)\n        missing = [key for key in loading.get("missing_keys", []) if not key.endswith("position_ids")]\n        if missing or loading.get("mismatched_keys"):\n            raise ValueError(f"A complete CLIP image AND text checkpoint is required; missing weights: {missing[:8]}")\n        self.model = self.model.to(device).eval()\n        self.model.requires_grad_(False)\n        if (Path(path) / "preprocessor_config.json").is_file():\n            self.image_processor = CLIPImageProcessor.from_pretrained(path, local_files_only=True)\n            self.preprocessing_source = "checkpoint_preprocessor_config"\n        else:\n            size = self.model.config.vision_config.image_size\n            if size != 224 or self.model.config.vision_config.patch_size != 32:\n                raise ValueError("Missing CLIP image preprocessing config for a non-ViT-B/32 checkpoint")\n            self.image_processor = CLIPImageProcessor()\n            self.preprocessing_source = "transformers_4_49_standard_clip_vit_b32_defaults"\n            print("CLIP preprocessor_config.json absent: using recorded standard CLIP ViT-B/32 preprocessing defaults", flush=True)\n        prompts = [template.format(name=name.replace("_", " ").replace("-", " ")) for name in names\n                   for template in ("a photo of {name}", "a scene showing {name}")]\n        vectors = self.encode_text(prompts).reshape(len(names), 2, -1).mean(1)\n        self.class_vectors = torch.nn.functional.normalize(vectors, dim=1)\n\n    def encode_text(self, texts):\n        import torch\n        result = []\n        with torch.inference_mode():\n            for start in range(0, len(texts), self.text_batch_size):\n                tokens = self.tokenizer(texts[start:start + self.text_batch_size], padding=True,\n                    truncation=True, max_length=self.model.config.text_config.max_position_embeddings, return_tensors="pt")\n                tokens = {key: value.to(self.device) for key, value in tokens.items()}\n                with model_autocast(self.device):\n                    features = self.model.get_text_features(**tokens)\n                result.append(torch.nn.functional.normalize(features.float(), dim=1))\n        return torch.cat(result)\n\n    def score(self, images, text_groups):\n        import torch\n        unique = list(dict.fromkeys(text for group in text_groups for text in group if text.strip()))\n        if not unique:\n            return [{} for _ in images]\n        texts = self.encode_text(unique)\n        positions = {text: index for index, text in enumerate(unique)}\n        pixels = self.image_processor(images=images, return_tensors="pt")["pixel_values"].to(self.device)\n        with torch.inference_mode(), model_autocast(self.device):\n            features = self.model.get_image_features(pixel_values=pixels)\n        features = torch.nn.functional.normalize(features.float(), dim=1)\n        image_scores = features @ texts.T\n        class_scores, class_indices = (texts @ self.class_vectors.T).max(1)\n        image_scores, class_scores, class_indices = image_scores.cpu().tolist(), class_scores.cpu().tolist(), class_indices.cpu().tolist()\n        result = []\n        for i, group in enumerate(text_groups):\n            result.append({text: {"image_cosine": image_scores[i][positions[text]],\n                "class_cosine": class_scores[positions[text]],\n                "class_name": self.names[class_indices[positions[text]]]} for text in group if text.strip()})\n        return result\n\n\ndef validate_settings(settings):\n    for key in ("batch_size", "num_beams", "num_return_sequences", "max_new_tokens", "max_spans_per_hypothesis", "clip_text_batch_size"):\n        if type(settings[key]) is not int or settings[key] < 1:\n            raise ValueError(f"{key} must be a positive integer")\n    if settings["num_beams"] < 2 or settings["num_return_sequences"] > settings["num_beams"]:\n        raise ValueError("Use beam search with 1 <= num_return_sequences <= num_beams and num_beams >= 2")\n\n\ndef cache_path(output, image_id):\n    return Path(output) / "image_cache" / (object_hash(image_id) + ".json")\n\n\ndef read_cached(output, record, identity):\n    path = cache_path(output, record["image_id"])\n    if not path.is_file():\n        return None\n    cached = json.loads(path.read_text(encoding="utf-8"))\n    if cached["identity"] != identity or cached["image_id"] != record["image_id"] or cached["split"] != record["split"]:\n        raise ValueError("Wrong experiment/image in cache; original cache was not modified")\n    if set(cached["rows"]) != set(METHODS):\n        raise ValueError("Incomplete per-image transaction in cache")\n    for method, row in cached["rows"].items():\n        if (set(row) != set(COLUMNS) or row["image_id"] != record["image_id"] or row["split"] != record["split"]\n                or row["captioner_fingerprint"] != identity or row["generation_method"] != method):\n            raise ValueError("Invalid cached row schema/identity")\n    return cached\n\n\ndef run_manifest(payload):\n    import torch\n    import transformers\n    validate_settings(payload["settings"])\n    records = payload["records"]\n    if not records or len({r["image_id"] for r in records}) != len(records):\n        raise ValueError("Missing/duplicate dataset records")\n    return {"version": METHOD_VERSION, "task": TASK, "settings": payload["settings"], "seed": payload["seed"],\n        "code_sha256": payload["code_sha256"], "dtd_split": payload["dtd_split"],\n        "class_names_for_post_generation_audit": payload["class_names"],\n        "base_florence_assets": checkpoint_asset_identity(payload["florence"]),\n        "clip_assets": checkpoint_asset_identity(payload["clip"]),\n        "image_preprocessing": "checkpoint_config_if_present_else_standard_clip_vit_b32_defaults",\n        "images": [{"image_id": r["image_id"], "split": r["split"], "sha256": file_sha256(r["image_path"])} for r in records],\n        "runtime": {"torch": torch.__version__, "transformers": transformers.__version__,\n            "bf16": torch.cuda.is_available() and torch.cuda.is_bf16_supported()},\n        "canonical_policy": "dtd_lexical_v2", "semantic_audit_threshold": .30,\n        "semantic_hard_gate": False, "trainable_parameters": 0}\n\n\ndef export_cached(payload, identity):\n    output = Path(payload["output"])\n    cached = [read_cached(output, r, identity) for r in payload["records"]]\n    complete = all(item is not None for item in cached)\n    available = [item for item in cached if item is not None]\n    folder = output if complete else output / "preview_incomplete"\n    paths = {}\n    for method in METHODS:\n        path = folder / "exports" / "captions.csv" if method == "multibeam_grounded" else folder / "controls" / method / "captions.csv"\n        atomic_csv(path, [item["rows"][method] for item in available], COLUMNS)\n        paths[method] = str(path)\n    audit_path = folder / "candidate_audit.jsonl"\n    audit_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = audit_path.with_suffix(".jsonl.tmp")\n    with temporary.open("w", encoding="utf-8") as handle:\n        for item in available:\n            handle.write(json.dumps({"image_id": item["image_id"], "split": item["split"],\n                "raw_proposals": item["proposals"], "candidates": item["candidate_audit"]}, ensure_ascii=False, allow_nan=False) + "\\n")\n    temporary.replace(audit_path)\n    summaries = []\n    for method in METHODS:\n        for role in ("train", "val", "test"):\n            rows = [item["rows"][method] for item in available if item["split"] == role]\n            if not rows:\n                continue\n            summaries.append({"method": method, "split": role, "images": len(rows),\n                "valid": sum(row["caption_valid"] for row in rows),\n                "valid_rate": sum(row["caption_valid"] for row in rows) / len(rows),\n                "unique_captions": len({row["final_caption"] for row in rows}),\n                "canonical_hits": sum(row["lexical_leakage_detected"] for row in rows),\n                "alias_hits_diagnostic": sum(row["alias_hit_diagnostic"] for row in rows),\n                "fallbacks": sum(row["selection_fallback"] for row in rows),\n                "mean_image_text_score": sum(row["image_text_similarity"] for row in rows if row["image_text_similarity"] != "") /\n                    max(1, sum(row["image_text_similarity"] != "" for row in rows))})\n    if summaries:\n        atomic_csv(folder / "caption_generation_summary.csv", summaries, list(summaries[0]))\n    summary = {"identity": identity, "complete": complete, "rows": len(available), "expected": len(cached),\n        "csv_paths": paths, "csv_sha256": {key: file_sha256(path) for key, path in paths.items()},\n        "optimized_clip_score_is_not_independent_quality_evidence": True,\n        "downstream_improvement_verified": False, "human_ground_truth": False}\n    atomic_json(output / "generation_status.json", summary)\n    return summary\n\n\ndef run_generation(payload, device, florence_factory=FrozenFlorence, ranker_factory=ImageTextRanker):\n    import torch\n    output = Path(payload["output"])\n    manifest = run_manifest(payload)\n    identity = object_hash(manifest)\n    manifest_path = output / "generation_manifest.json"\n    if manifest_path.is_file() and json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:\n        raise ValueError("Changed assets/code/dataset/settings/runtime: choose a new output directory")\n    if not manifest_path.is_file() and ((output / "image_cache").exists() or (output / "exports").exists()):\n        raise ValueError("Existing generated artifacts have no matching manifest; use another output directory")\n    atomic_json(manifest_path, manifest)\n    pending = [r for r in payload["records"] if read_cached(output, r, identity) is None]\n    limit = payload.get("max_new_images", 0)\n    if limit < 0:\n        raise ValueError("max_new_images must be >= 0; zero processes all missing images")\n    if limit:\n        pending = pending[:limit]\n    print(f"Grounded captions: remaining this run={len(pending)}; output={output}", flush=True)\n    if pending:\n        generator = florence_factory(payload["florence"], device)\n        ranker = ranker_factory(payload["clip"], payload["class_names"], device, payload["settings"]["clip_text_batch_size"])\n        atomic_json(output / "clip_preprocessing_used.json", {"source": ranker.preprocessing_source,\n                    "config": ranker.image_processor.to_dict()})\n        size, offset = payload["settings"]["batch_size"], 0\n        while offset < len(pending):\n            group = pending[offset:offset + size]\n            try:\n                images = [load_rgb(r["image_path"]) for r in group]\n                greedy = generator.generate(images, 1, 1, payload["settings"]["max_new_tokens"])\n                beams = generator.generate(images, payload["settings"]["num_beams"], payload["settings"]["num_return_sequences"], payload["settings"]["max_new_tokens"])\n                if (len(greedy) != len(group) or len(beams) != len(group) or any(len(p) != 1 for p in greedy)\n                        or any(len(p) != payload["settings"]["num_return_sequences"] for p in beams)):\n                    raise ValueError("Wrong image/hypothesis count from Florence")\n                proposals = [one + many for one, many in zip(greedy, beams)]\n                text_groups = []\n                for raw in proposals:\n                    prefix, single, multiple = proposal_sets(raw, payload["class_names"], payload["settings"]["max_spans_per_hypothesis"])\n                    text_groups.append(list(dict.fromkeys([prefix["text"]] + [item["text"] for item in multiple])))\n                scores = ranker.score(images, text_groups)\n                if len(scores) != len(group) or any(set(score) != {t for t in texts if t.strip()} for score, texts in zip(scores, text_groups)):\n                    raise ValueError("Wrong CLIP score count/keys; no batch rows were cached")\n                if any(not math.isfinite(value["image_cosine"]) or not math.isfinite(value["class_cosine"])\n                       for score in scores for value in score.values()):\n                    raise ValueError("Non-finite CLIP score")\n                transactions = [build_image_result(r, p, s, payload["class_names"], payload["settings"], identity)\n                                for r, p, s in zip(group, proposals, scores)]\n            except torch.cuda.OutOfMemoryError:\n                if len(group) == 1:\n                    raise\n                size = max(1, len(group) // 2)\n                torch.cuda.empty_cache()\n                print("Caption batch reduced after CUDA OOM to", size, flush=True)\n                continue\n            for record, result in zip(group, transactions):\n                atomic_json(cache_path(output, record["image_id"]), result)\n            offset += len(group)\n            print(f"Completed {offset}/{len(pending)} newly queued images", flush=True)\n    return export_cached(payload, identity)\n\n\ndef worker_main(payload_path):\n    import torch\n    import transformers\n    if not transformers.__version__.startswith("4.49."):\n        raise RuntimeError("Use the isolated Transformers 4.49.x worker for the attached legacy Florence model")\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a GPU accelerator for this Florence + full CLIP generation worker")\n    payload = json.loads(Path(payload_path).read_text(encoding="utf-8"))\n    torch.manual_seed(payload["seed"])\n    torch.cuda.manual_seed_all(payload["seed"])\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    summary = run_generation(payload, torch.device("cuda"))\n    print(json.dumps(summary, indent=2), flush=True)\n\n\ndef effective_dtd_v2(row, names):\n    text = clean_dtd_caption(str(row.get("raw_caption", "")).strip() or row["final_caption"])\n    try:\n        finite_score = math.isfinite(float(row["semantic_leakage_score"]))\n    except (TypeError, ValueError):\n        finite_score = False\n    recoverable = bool_value(row["caption_valid"]) or (bool_value(row["semantic_leakage_detected"]) and finite_score)\n    return text, bool(valid_short(text, names) and recoverable)\n\n\ndef compare_caption_files(paths, records, names, output_dir, write_matched_masks=True):\n    """Read existing CSVs without changing them; compare the actual dtd_lexical_v2 model inputs."""\n    expected = {r["image_id"]: r["split"] for r in records}\n    if not paths or len(expected) != len(records):\n        raise ValueError("Provide named complete CSVs and distinct dataset records")\n    methods, source_hashes = {}, {}\n    for label, path in paths.items():\n        if not re.fullmatch(r"[a-z0-9_]+", label):\n            raise ValueError("Comparison labels must be lowercase letters/digits/underscores")\n        rows = read_csv_rows(path, ["image_id", "split", "final_caption", "caption_valid", "leakage_detected",\n                                    "semantic_leakage_detected", "semantic_leakage_score"])\n        mapped = {row["image_id"]: row for row in rows}\n        if len(mapped) != len(rows) or set(mapped) != set(expected) or any(row["split"] != expected[key] for key, row in mapped.items()):\n            raise ValueError(f"{label}: incomplete, duplicate or wrong official DTD split; original CSV was not modified")\n        methods[label] = {key: {"row": mapped[key], "text": effective_dtd_v2(mapped[key], names)[0],\n                                "valid": effective_dtd_v2(mapped[key], names)[1]} for key in expected}\n        source_hashes[label] = file_sha256(path)\n    output_dir = Path(output_dir)\n    signatures = {"csv_sha256": source_hashes, "policy": "dtd_lexical_v2", "names": names,\n                  "image_ids_and_splits": expected, "comparison_version": 1}\n    lock = output_dir / "comparison_manifest.json"\n    if lock.exists() and json.loads(lock.read_text(encoding="utf-8")) != signatures:\n        raise ValueError("Comparison inputs changed; choose a new comparison output directory")\n    atomic_json(lock, signatures)\n    common = {key: all(method[key]["valid"] for method in methods.values()) for key in expected}\n    summaries, details = [], []\n    first_label = next(iter(methods))\n    for label, method in methods.items():\n        for role in ("train", "val", "test"):\n            entries = [entry for key, entry in method.items() if expected[key] == role]\n            if entries:\n                summaries.append({"method": label, "split": role, "images": len(entries),\n                    "valid": sum(entry["valid"] for entry in entries),\n                    "valid_rate": sum(entry["valid"] for entry in entries) / len(entries),\n                    "unique_input_texts": len({entry["text"] for entry in entries}),\n                    "valid_with_alias_hit": sum(entry["valid"] and bool(term_hits(entry["text"], names, True)) for entry in entries),\n                    "common_valid": sum(common[key] for key in expected if expected[key] == role),\n                    "text_changed_vs_first": sum(method[key]["text"] != methods[first_label][key]["text"] for key in expected if expected[key] == role)})\n        for key, entry in method.items():\n            details.append({"method": label, "image_id": key, "split": expected[key], "model_input_text": entry["text"],\n                            "valid": entry["valid"], "common_valid": common[key]})\n        if write_matched_masks:\n            masked = []\n            for record in records:\n                key = record["image_id"]\n                row = dict(method[key]["row"])\n                row.update(original_caption_valid=row["caption_valid"], original_semantic_flag=row["semantic_leakage_detected"],\n                    original_semantic_score=row["semantic_leakage_score"], caption_valid=common[key],\n                    caption_input_policy="dtd_lexical_v2", comparison_mask="common_valid_intersection")\n                if not common[key]:\n                    # Prevent legacy semantic-only recovery from switching a deliberately masked caption back on.\n                    row.update(semantic_leakage_detected=False, semantic_leakage_score="",\n                               semantic_audit_status="original_audit_retained_in_original_columns")\n                masked.append(row)\n            columns = list(dict.fromkeys(key for row in masked for key in row))\n            atomic_csv(output_dir / "matched_masks" / label / "captions.csv", masked, columns)\n            assert all(effective_dtd_v2(row, names)[1] == common[row["image_id"]] for row in masked)\n    atomic_csv(output_dir / "caption_comparison_summary.csv", summaries, list(summaries[0]))\n    atomic_csv(output_dir / "caption_comparison_details.csv", details, list(details[0]))\n    atomic_csv(output_dir / "common_valid_mask.csv", [{"image_id": key, "split": role, "caption_valid": common[key]}\n               for key, role in expected.items()], ["image_id", "split", "caption_valid"])\n    return summaries\n\n\nif __name__ == "__main__":\n    worker_main(sys.argv[1])\n'


In [ ]:
model_dir, backend = florence_checkpoint(FLORENCE_MODEL_REF, [Path(OFFLINE_ASSETS_ROOT), Path("/kaggle/input")])
if backend != "legacy":
    raise ValueError("Use the attached legacy Microsoft Florence-2-base; this is not the LoRA adapter or native-community experiment")
clip_dir = Path(CLIP_MODEL_LOCAL_PATH).expanduser()
if not (clip_dir / "config.json").is_file():
    raise FileNotFoundError(f"Set CLIP_MODEL_LOCAL_PATH to the complete local CLIP ViT-B/32 folder: {clip_dir}")
if json.loads((clip_dir / "config.json").read_text(encoding="utf-8")).get("model_type") != "clip":
    raise ValueError("The supplied folder is not a full CLIP checkpoint")
RUNTIME_DIR = OUTPUT_DIR / "caption_runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
worker_env = caption_runtime_environment("legacy", OFFLINE_ASSETS_ROOT, RUNTIME_DIR)
payload = {"output": str(OUTPUT_DIR.resolve()), "records": caption_records, "dtd_split": DTD_SPLIT,
    "class_names": class_names, "florence": str(model_dir), "clip": str(clip_dir.resolve()),
    "settings": SETTINGS, "seed": SEED, "max_new_images": MAX_NEW_IMAGES,
    "code_sha256": object_hash(GROUNDED_WORKER_SOURCE)}
payload["settings"] = dict(SETTINGS, seed=SEED)
PAYLOAD_PATH = RUNTIME_DIR / "request.json"
atomic_json(PAYLOAD_PATH, payload)
run_grounded_worker(GROUNDED_WORKER_SOURCE, PAYLOAD_PATH, RUNTIME_DIR, worker_env)
generation_status = json.loads((OUTPUT_DIR / "generation_status.json").read_text(encoding="utf-8"))
print("Complete:", generation_status["complete"], "| cached images:", generation_status["rows"])
if not generation_status["complete"]:
    print("SMOKE TEST ONLY: preview CSVs are incomplete. Set MAX_NEW_IMAGES=0 and rerun this cell to resume.")


## Inspect Caption Content and Coverage

`raw_caption` contains the actual selected short text, because M-TACS reads raw_caption before final_caption.
The unabridged Florence output is preserved in `source_long_caption`; every chosen span has its hypothesis,
sentence and word interval in the cleaned source sentence. `candidate_audit.jsonl` retains proposals and scores.
An interval of -1 denotes the original-prefix control/fallback rather than an extracted sentence span.

`image_text_similarity` ranks proposals against their image. `semantic_leakage_score` compares the selected text
to class prompts for audit only. They are distinct scores. CLIP alignment is optimized by this method, so a higher
score is **not independent evidence** of better captions or increased M-TACS F1. Check visible details yourself.


In [ ]:
import pandas as pd
from IPython.display import display

report_dir = OUTPUT_DIR if generation_status["complete"] else OUTPUT_DIR / "preview_incomplete"
summary = pd.read_csv(report_dir / "caption_generation_summary.csv")
display(summary)
main_csv = Path(generation_status["csv_paths"]["multibeam_grounded"])
main_rows = read_csv_rows(main_csv)
display(pd.DataFrame(main_rows)[["split", "source_long_caption", "final_caption", "caption_valid",
                                "image_text_similarity", "alias_hit_diagnostic", "selection_fallback"]].head(10))
print("Main CSV:", main_csv)
print("These are generated captions, not reference annotations or human ground truth.")


In [ ]:
def show_train_caption(index=0):
    train_records = [row for row in caption_records if row["split"] == "train"]
    if not 0 <= index < len(train_records):
        raise IndexError("Choose an official training-image index")
    record = train_records[index]
    cached = read_cached(OUTPUT_DIR, record, generation_status["identity"])
    if cached is None:
        print("This image is not generated yet; finish/resume generation first.")
        return
    image = load_rgb(record["image_path"])
    image.thumbnail((650, 450))
    display(image)
    print("Training review index:", index)
    for method in METHODS:
        row = cached["rows"][method]
        print(method, "|", row["final_caption"], "| valid:", row["caption_valid"])

show_train_caption(0)  # Change the index for image-grounded review; category labels are not displayed.


## Compare to the Two Previous CSVs

Set the exact paths in `OLD_CAPTIONS_CSV` and `LORA_CAPTIONS_CSV`, then rerun this section.
Blank paths are skipped. Wrong, duplicate or incomplete DTD files stop with an error. Originals are read-only.
Reports compare the actual **dtd_lexical_v2 model inputs**, including legacy semantic-only reject recovery;
do not confuse a stored caption_valid column in an older schema with its effective M-TACS mask.

The matched-mask CSVs keep every image but disable text on the **same intersection** for all compared methods.
Original audit fields are retained separately. They are for screened/full_both experiments, not unscreened/redacted
variants which intentionally use different masks. Use separate M-TACS runs for natural masks vs matched masks.
Choose which comparisons to include before training; adding a CSV changes the intersection and requires a new
comparison output directory. High coverage, lexical cleanliness and CLIP similarity do not prove visual truth.


In [ ]:
if not generation_status["complete"]:
    print("Skipping complete-dataset comparison: finish the smoke test/resume first.")
else:
    comparisons = {}
    if OLD_CAPTIONS_CSV:
        comparisons["original_old"] = OLD_CAPTIONS_CSV
    if LORA_CAPTIONS_CSV:
        comparisons["decoder_lora"] = LORA_CAPTIONS_CSV
    comparisons.update({method: path for method, path in generation_status["csv_paths"].items()})
    # A path-set-specific folder avoids accidentally mixing intersections from different comparison sets.
    comparison_tag = object_hash({key: file_sha256(path) for key, path in comparisons.items()})[:12]
    COMPARISON_DIR = OUTPUT_DIR / "comparisons" / comparison_tag
    comparison = compare_caption_files(comparisons, caption_records, class_names, COMPARISON_DIR, WRITE_MATCHED_MASK_CSVS)
    display(pd.DataFrame(comparison))
    print("Comparison reports:", COMPARISON_DIR)
    if WRITE_MATCHED_MASK_CSVS:
        print("Matched-mask CSVs:", COMPARISON_DIR / "matched_masks")


## Downstream M-TACS Evaluation

Attach the completed main `exports/captions.csv` to **notebook 05 full_both**, train a fresh classifier and text cache,
and keep DTD split 1, seed/pool seed 42, batch 32, K=1 and 50 epochs. Set `GENERATE_CAPTIONS_IF_MISSING=False`;
do not let a classifier notebook regenerate missing rows using the old captioner. Use a new output directory.
Leave `CAPTION_INPUT_POLICY="dtd_lexical_v2"`. Select the classifier checkpoint by validation loss as before.

Historical single-seed test macro-F1 references: original full_both 0.809124; LoRA-caption full_both 0.806707;
visual_only 0.848168. These are observations from prior runs, not goals used to tune this caption selector.
The new method has **no measured F1 yet**. Higher F1 than the old full_both is an improvement for that run;
it is not automatically a robust gain or a win over visual_only. Confirm with paired seeds and locked evaluation.

To explain a difference, compare first-valid vs single-grounded, single-grounded vs main, caption validity,
alias hits, fallbacks, selected contexts and query/candidate gates. The common-mask CSVs help separate text-content
effects from changed availability; they do not hold the selected context fixed when the semantic selector is active.
No automatic test-score search or claim of guaranteed improvement is made.

Example additions to notebook 05's configuration cell (replace the attached CSV path):

```python
CAPTIONS_INPUT_PATH = "/kaggle/input/YOUR_ATTACHED_DATASET/captions.csv"
CAPTION_INPUT_POLICY = "dtd_lexical_v2"
GENERATE_CAPTIONS_IF_MISSING = False
os.environ["DTD_FULL_BOTH_OUTPUT"] = "/kaggle/working/outputs/dtd_full_both_grounded_seed42"
```

Sources: [Microsoft Florence-2 model](https://huggingface.co/microsoft/Florence-2-base),
[Transformers 4.49 generation parameters](https://huggingface.co/docs/transformers/v4.49.0/en/main_classes/text_generation),
[CLIP image/text feature APIs](https://huggingface.co/docs/transformers/v4.49.0/en/model_doc/clip).
